# Nimbus Aspirate and Dispense Demo

This notebook demonstrates aspirate and dispense operations with the Hamilton Nimbus backend.

The demo covers:
1. Creating a Nimbus Deck and assigning resources
2. Setting up the NimbusBackend and LiquidHandler
3. Picking up tips from the tip rack
4. Aspirating 50 µL from wells (2mm above bottom)
5. Dispensing to wells (2mm above bottom)
6. Dropping tips to waste
7. Cleaning up and closing the connection


## Setup


In [1]:
# Import necessary modules
import sys
import logging

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends.hamilton.nimbus_backend import NimbusBackend
from pylabrobot.resources.hamilton.nimbus_decks import NimbusDeck
from pylabrobot.resources import (
  hamilton_96_tiprack_300uL_filter,
  Cor_Axy_96_wellplate_500uL_Ub,
  Coordinate,
)
from pylabrobot.visualizer import Visualizer


# Setup logging
plr_logger = logging.getLogger('pylabrobot')
plr_logger.setLevel(logging.INFO)  # INFO for normal use, DEBUG for troubleshooting
plr_logger.handlers.clear()
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter('%(levelname)s - %(message)s'))
plr_logger.addHandler(console_handler)

# ========================================================================
# CREATE DECK AND RESOURCES (using coordinates from nimbus_deck_setup.ipynb)
# ========================================================================

# Create NimbusDeck using default values (layout 8 dimensions)
deck = NimbusDeck()

print(f"Deck created: {deck.name}")
print(f"  Size: {deck.get_size_x()} x {deck.get_size_y()} x {deck.get_size_z()} mm")
print(f"  Rails: {deck.num_rails}")

tip_rack = hamilton_96_tiprack_300uL_filter(name="Tips", with_tips=True)
deck.assign_child_resource(tip_rack, location=Coordinate(x=305.750, y=126.537, z=128.620))

print(f"\nTip rack assigned: {tip_rack.name}")
wellplate = Cor_Axy_96_wellplate_500uL_Ub(name="Plate", with_lid=False)
deck.assign_child_resource(wellplate, location=Coordinate(x=438.070, y=124.837, z=101.490))

print(f"Wellplate assigned: {wellplate.name}")
print(f"  Waste block: {deck.get_resource('default_long_block').name}")

visualizer = Visualizer(deck, open_browser=False)
await visualizer.setup()

Deck created: deck
  Size: 831.85 x 424.18 x 300.0 mm
  Rails: 30

Tip rack assigned: Tips
Wellplate assigned: Plate
  Waste block: default_long_block
Websocket server started at http://127.0.0.1:2121
File server started at http://127.0.0.1:1337 . Open this URL in your browser.


In [2]:
# Create NimbusBackend instance
# Replace with your instrument's IP address
backend = NimbusBackend(host="192.168.100.100", port=2000)
# Create LiquidHandler with backend and deck
lh = LiquidHandler(backend=backend, deck=deck)
print("LiquidHandler created successfully")

# Setup
await lh.setup(unlock_door=False)
print("\n" + "="*60)
print("SETUP COMPLETE")
print("="*60)
print(f"Setup finished: {backend.setup_finished}")
print(f"\nInstrument Configuration:")
print(f"  Number of channels: {backend.num_channels}")


LiquidHandler created successfully
INFO - Connection initialized (Client ID: 3, Address: 2:3:65535)
INFO - Client registered.
INFO - Setup complete. Registered as Client ID 3 (2:3:65535), Root: NimbusCORE
INFO - Interfaces: door_lock, nimbus_core, pipette
INFO - Channel configuration: 4 channels
INFO - Tip presence: [False, False, False, False]
INFO - Instrument initialized: True
INFO - Door already locked
INFO - Instrument already initialized, skipping initialization

SETUP COMPLETE
Setup finished: True

Instrument Configuration:
  Number of channels: 4


## Define Resources

In [3]:
# Resources are already created in the setup cell above
# tip_rack and wellplate variables are available

print(f"Tip rack: {tip_rack.name} ({tip_rack.num_items} tips)")
print(f"Source/Destination plate: {wellplate.name} (using same plate, different wells)")

# Use wellplate as both source and destination
source_plate = wellplate
destination_plate = wellplate

# Get waste positions
waste_block = deck.get_resource("default_long_block")
waste_positions = waste_block.children[:4]

print(f"Waste positions: {[wp.name for wp in waste_positions]}")


Tip rack: Tips (96 tips)
Source/Destination plate: Plate (using same plate, different wells)
Waste positions: ['default_long_1', 'default_long_2', 'default_long_3', 'default_long_4']


## Pick Up Tips

Pick up tips from positions A1-D1.


In [4]:
plr_logger.setLevel(logging.DEBUG)  # INFO for normal use, DEBUG for troubleshooting
# Get the first 4 tip spots (A1, B1, C1, D1)
tip_spots = tip_rack['A1:D1']

print(f"Picking up tips from positions: {[ts.get_identifier() for ts in tip_spots]}")
await lh.pick_up_tips(tip_spots)
print("✓ Tips picked up successfully!")


Picking up tips from positions: ['A1', 'B1', 'C1', 'D1']
DEBUG - pick_up_tips(tip_spots=['Tips_tipspot_A1', 'Tips_tipspot_B1', 'Tips_tipspot_C1', 'Tips_tipspot_D1'], use_channels=None, offsets=None)
DEBUG - IsTipPresent parameters: {}
DEBUG - PickupTips parameters: {'channels_involved': [1, 1, 1, 1], 'x_positions': [16144, 16144, 16144, 16144], 'y_positions': [-16899, -17799, -18699, -19599], 'minimum_traverse_height_at_beginning_of_a_command': 14600, 'begin_tip_pick_up_process': [13802, 13802, 13802, 13802], 'end_tip_pick_up_process': [13002, 13002, 13002, 13002], 'tip_types': [<NimbusTipType.STANDARD_300UL_FILTER: 1>, <NimbusTipType.STANDARD_300UL_FILTER: 1>, <NimbusTipType.STANDARD_300UL_FILTER: 1>, <NimbusTipType.STANDARD_300UL_FILTER: 1>]}
INFO - Picked up tips on channels [0, 1, 2, 3]
✓ Tips picked up successfully!


## Aspirate Operation

Aspirate 50 µL from wells A1-D1, 2mm above the bottom of the well.


In [5]:
# Get source wells (A1, B1, C1, D1)
source_wells = source_plate['A1:D1']

print(f"Aspirating 50 µL from wells: {[w.get_identifier() for w in source_wells]}")
print(f"  Liquid height: 2.0 mm above bottom")

# Aspirate with liquid_height=2.0mm
# Tips are already picked up, so LiquidHandler will use them automatically
await lh.aspirate(
    source_wells,
    vols = 4 *[50],  # Can be a single number (applies to all channels) or a list
    liquid_height = 4 *[2],  # 2mm above bottom of well (can be a single float or list)
    flow_rates = 4 *[100],
)

print("✓ Aspiration complete!")


Aspirating 50 µL from wells: ['A1', 'B1', 'C1', 'D1']
  Liquid height: 2.0 mm above bottom
DEBUG - aspirate(resources=['Plate_well_A1', 'Plate_well_B1', 'Plate_well_C1', 'Plate_well_D1'], vols=[50, 50, 50, 50], use_channels=None, flow_rates=[100, 100, 100, 100], offsets=None, liquid_height=[2, 2, 2, 2], blow_out_air_volume=None)
DEBUG - DisableADC parameters: {'channels_involved': [1, 1, 1, 1]}
INFO - Disabled ADC before aspirate
DEBUG - GetChannelConfiguration parameters: {'channel': 1, 'indexes': [2]}
DEBUG - Channel 1 configuration (index 2): enabled=False
DEBUG - GetChannelConfiguration parameters: {'channel': 2, 'indexes': [2]}
DEBUG - Channel 2 configuration (index 2): enabled=False
DEBUG - GetChannelConfiguration parameters: {'channel': 3, 'indexes': [2]}
DEBUG - Channel 3 configuration (index 2): enabled=False
DEBUG - GetChannelConfiguration parameters: {'channel': 4, 'indexes': [2]}
DEBUG - Channel 4 configuration (index 2): enabled=False
DEBUG - Aspirate parameters: {'aspirat

## Dispense Operation

Dispense 50 µL to wells A2-D2, 2mm above the bottom of the well.


In [6]:
# Get destination wells (A2, B2, C2, D2)
dest_wells = destination_plate['A7:D7']

print(f"Dispensing 50 µL to wells: {[w.get_identifier() for w in dest_wells]}")
print(f"  Liquid height: 2.0 mm above bottom")

# Dispense with liquid_height=2.0mm
# Tips are already picked up, so LiquidHandler will use them automatically
await lh.dispense(
    dest_wells,
    vols= 4 *[50],  # Can be a single number (applies to all channels) or a list
    liquid_height= 4 *[2],  # 2mm above bottom of well (can be a single float or list)
    flow_rates= 4 *[120],
)

print("✓ Dispense complete!")


Dispensing 50 µL to wells: ['A7', 'B7', 'C7', 'D7']
  Liquid height: 2.0 mm above bottom
DEBUG - dispense(resources=['Plate_well_A7', 'Plate_well_B7', 'Plate_well_C7', 'Plate_well_D7'], vols=[50, 50, 50, 50], use_channels=None, flow_rates=[120, 120, 120, 120], offsets=None, liquid_height=[2, 2, 2, 2], blow_out_air_volume=None)
DEBUG - DisableADC parameters: {'channels_involved': [1, 1, 1, 1]}
INFO - Disabled ADC before dispense
DEBUG - GetChannelConfiguration parameters: {'channel': 1, 'indexes': [2]}
DEBUG - Channel 1 configuration (index 2): enabled=False
DEBUG - GetChannelConfiguration parameters: {'channel': 2, 'indexes': [2]}
DEBUG - Channel 2 configuration (index 2): enabled=False
DEBUG - GetChannelConfiguration parameters: {'channel': 3, 'indexes': [2]}
DEBUG - Channel 3 configuration (index 2): enabled=False
DEBUG - GetChannelConfiguration parameters: {'channel': 4, 'indexes': [2]}
DEBUG - Channel 4 configuration (index 2): enabled=False
DEBUG - Dispense parameters: {'dispense_

## Drop Tips

Drop tips to waste positions.


In [8]:
print(f"Dropping tips at waste positions: {[wp.name for wp in waste_positions]}")
await lh.drop_tips(waste_positions)

print("✓ Tips dropped successfully!")


Dropping tips at waste positions: ['default_long_1', 'default_long_2', 'default_long_3', 'default_long_4']
DEBUG - drop_tips(tip_spots=['default_long_1', 'default_long_2', 'default_long_3', 'default_long_4'], use_channels=None, offsets=None, allow_nonzero_volume=False)
DEBUG - DropTipsRoll parameters: {'channels_involved': [1, 1, 1, 1], 'x_positions': [55375, 55375, 55375, 55375], 'y_positions': [1986, 188, -7615, -9413], 'minimum_traverse_height_at_beginning_of_a_command': 14600, 'begin_tip_deposit_process': [13539, 13539, 13539, 13539], 'end_tip_deposit_process': [13139, 13139, 13139, 13139], 'z_position_at_end_of_a_command': [14600, 14600, 14600, 14600], 'roll_distances': [900, 900, 900, 900]}
INFO - Dropped tips on channels [0, 1, 2, 3]
✓ Tips dropped successfully!


## Cleanup

Finally, we'll stop the liquid handler and close the connection.


In [9]:
plr_logger.setLevel(logging.INFO)  # INFO for normal use, DEBUG for troubleshooting

# Stop and close connection
await lh.backend.park()
await lh.backend.unlock_door()
await lh.stop()

print("Connection closed successfully")


INFO - Instrument parked successfully
INFO - Door unlocked successfully
INFO - Closing connection to socket 192.168.100.100:2000
INFO - Hamilton TCP client stopped
Connection closed successfully
